# Lecture 07 — CNN Physics Applications: Real Data & Transfer Learning

**PHYG004 — Machine Learning Applications in Physics, 2026 Spring · Sogang University**
Prof. Young Woo Choi

---

## What you'll do today

We take the convolutional networks (CNNs) you built on a *toy* physics
dataset in Lecture 06 and point them at **two research problems**, reproducing
the core results of two influential papers:

| Mission | Paper | Physics question |
|---|---|---|
| **A** | Carrasquilla & Melko, *Nat. Phys.* **13**, 431 (2017) | Can a neural net *find* a phase transition it was never told the location of? |
| **B** | Dieleman, Willett & Dambre, *MNRAS* **450**, 1441 (2015) | Can a CNN classify galaxy morphology from real telescope images? |
| **B+** | (new) transfer learning | Can an ImageNet-pretrained network reach the same accuracy **faster**? |

By the end you will be able to:

1. Run a Metropolis Monte-Carlo simulation of the 2D Ising model and **generate labeled data**.
2. Train an **FNN** and a **CNN** to classify ordered vs. disordered spin configurations, and **estimate $T_c$** from the network output — comparing to the exact Onsager value.
3. Classify **real DECam galaxy images** (Galaxy10 DECals) into elliptical vs. spiral with a from-scratch CNN.
4. **Fine-tune an ImageNet-pretrained ResNet-18** on the same task (freeze the backbone, replace the head) and benchmark accuracy *and* training time against the from-scratch model.

> **Framework note.** This notebook uses **PyTorch** (not JAX). Missions B/B+
> rely on `torchvision`'s pretrained ResNet-18, which is the standard route for
> transfer learning, so we stay in the PyTorch ecosystem throughout.

> **Runtime.** Everything runs on a **free Colab CPU**: Ising data ~1–2 min,
> Ising training seconds, Galaxy CNN ~3 min, ResNet-18 fine-tune ~2 min
> (frozen backbone means no backbone gradients). No GPU required, though a GPU
> makes Mission B/B+ snappier.


## 0. Where this sits in the course — and a data label

> ### Data label: **Real measured data** (DECam observed images)
>
> In **Lecture 06** the CNN you built only ever touched a **toy** dataset
> (synthetic 2D Ising configurations you generated yourself). In **this**
> session the CNN reaches **real observed images for the first time**:
> Mission B uses **Galaxy10 DECals**, actual galaxy cutouts photographed by the
> **Dark Energy Camera (DECam)**.

Throughout the course we label every dataset by *where it comes from*:

| Label | Meaning | Example |
|---|---|---|
| **Toy / synthetic** | We generated it from a known model | Mission A: Ising Monte-Carlo configs |
| **Real measured data** | A physical instrument recorded it | Mission B: DECam galaxy images |

Mission A is still synthetic — but it is a *physicist's* synthetic data: every
configuration is a genuine Boltzmann sample from $e^{-\beta H}$, and the label
("ordered" vs. "disordered") is grounded in the **exact** Onsager solution. It
is the perfect controlled warm-up before we face the noise of real telescope
data in Mission B.


## 1. Setup

First cell installs dependencies (uncomment on Colab). Second cell does imports,
sets the global seed, and checks the device.

In [ ]:
# On Google Colab, uncomment the next line. torch / torchvision / numpy /
# matplotlib are pre-installed on Colab; numba, gdown, h5py may need installing.
# !pip install -q torch torchvision numpy matplotlib scikit-learn tqdm numba gdown h5py Pillow


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib.pyplot as plt

# Reproducibility — single global seed for the whole notebook.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")


---

# Mission A — Detecting a phase transition with a neural network

**Paper:** Carrasquilla & Melko, *Machine learning phases of matter*,
*Nature Physics* **13**, 431 (2017).

## The physics first

The **2D Ising model** is *the* textbook model of a continuous phase transition.
Spins $s_i = \pm 1$ live on a square lattice and interact with their nearest
neighbors:

$$ H = -J \sum_{\langle i,j\rangle} s_i s_j. $$

At low temperature the system is **ordered** (ferromagnetic — most spins
aligned, $|M|\to 1$); at high temperature it is **disordered** (paramagnetic —
spins random, $|M|\to 0$). The two phases meet at the **critical temperature**,
known *exactly* from Onsager's 1944 solution:

$$ T_c = \frac{2}{\ln(1+\sqrt{2})} \approx 2.269 \; (J/k_B). $$

**The ML idea.** We *never tell the network where $T_c$ is.* We only label
configurations as "ordered" (sampled at $T<T_c$) or "disordered" ($T>T_c$),
train a classifier, then ask: at what temperature does the network flip its
mind, $P(\text{ordered})=0.5$? If the network has learned real physics, that
crossover should land on Onsager's $T_c$ — a transition it rediscovered on its
own.

> **Physicist's dictionary.** Loss $\leftrightarrow$ energy to minimize; the
> softmax output $P(\text{ordered})$ behaves like an **order parameter** that
> the network learns to read off from the raw spin texture.


## A.1 — Monte-Carlo engine (Metropolis–Hastings)

One **sweep** = $L^2$ attempted single-spin flips. The Metropolis rule: flip
always if $\Delta E \le 0$, otherwise flip with probability $e^{-\beta\Delta E}$,
where for a single spin $\Delta E = 2 s_i \sum_{\text{nn}} s_j$. We JIT-compile
the inner loop with **Numba** for a ~100× speedup over pure Python.

> Heavy-ish cell, but cheap (Numba compiles on first call). On Colab this runs
> in seconds.

In [ ]:
import numpy as np
from numba import njit


@njit
def _metropolis_sweep(lattice, L, beta, rand_i, rand_j, rand_u):
    """One JIT-compiled Metropolis sweep: L*L single-spin flip attempts."""
    for k in range(L * L):
        i = rand_i[k]
        j = rand_j[k]
        s = lattice[i, j]
        # Nearest neighbors with periodic boundary conditions
        nn_sum = (
            lattice[(i + 1) % L, j]
            + lattice[(i - 1) % L, j]
            + lattice[i, (j + 1) % L]
            + lattice[i, (j - 1) % L]
        )
        dE = 2.0 * s * nn_sum
        if dE <= 0 or rand_u[k] < np.exp(-beta * dE):
            lattice[i, j] = -s


@njit
def _compute_energy(lattice, L):
    energy = 0.0
    for i in range(L):
        for j in range(L):
            s = lattice[i, j]
            energy -= s * (lattice[(i + 1) % L, j] + lattice[i, (j + 1) % L])
    return energy


def initialize_lattice(L, random=True, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    if random:
        return rng.choice(np.array([-1, 1]), size=(L, L)).astype(np.int64)
    return np.ones((L, L), dtype=np.int64)


def compute_magnetization(lattice):
    """Magnetization per spin, |M| in [0, 1]."""
    return np.abs(np.mean(lattice))


def metropolis_sweep(lattice, T, rng):
    L = lattice.shape[0]
    beta = 1.0 / T
    n = L * L
    rand_i = rng.integers(0, L, size=n)
    rand_j = rng.integers(0, L, size=n)
    rand_u = rng.random(size=n)
    _metropolis_sweep(lattice, L, beta, rand_i, rand_j, rand_u)
    return lattice


def run_simulation(L, T, n_sweeps=1000, n_equilib=500, n_samples=1,
                   sample_interval=10, seed=None):
    """Run an Ising MC simulation at temperature T and collect configs."""
    rng = np.random.default_rng(seed)
    # Start ordered at low T, random at high T (faster equilibration)
    ordered_start = T < 2.269
    lattice = initialize_lattice(L, random=not ordered_start, rng=rng)

    for _ in range(n_equilib):          # equilibration
        metropolis_sweep(lattice, T, rng)

    configs, mags, energies = [], [], []
    for sweep in range(n_sweeps - n_equilib):   # measurement
        metropolis_sweep(lattice, T, rng)
        if sweep % sample_interval == 0 and len(configs) < n_samples:
            configs.append(lattice.copy())
            mags.append(compute_magnetization(lattice))
            energies.append(_compute_energy(lattice, L) / (L * L))
    return {"configs": configs, "magnetizations": mags, "energies": energies}


### Checkpoint A.1 — does the simulator know the physics?

Run a quick sanity check at three temperatures. **Expected** (Onsager $T_c\approx2.269$):

- $T=1.5$ (deep in ordered phase) $\Rightarrow |M| > 0.8$
- $T=2.27$ (near $T_c$) $\Rightarrow$ intermediate, fluctuating
- $T=3.5$ (disordered) $\Rightarrow |M| < 0.2$

In [ ]:
L = 16
print("Compiling Numba (first call is slow)...")
for T in [1.5, 2.269, 3.5]:
    res = run_simulation(L, T, n_sweeps=2000, n_equilib=1000, n_samples=1, seed=42)
    print(f"  T={T:5.3f}:  |M| = {res['magnetizations'][0]:.3f}   "
          f"E/N = {res['energies'][0]:.3f}")

# Verification of the checkpoint criteria
m_ord = run_simulation(16, 1.5, 2000, 1000, 1, seed=1)["magnetizations"][0]
m_dis = run_simulation(16, 3.5, 2000, 1000, 1, seed=1)["magnetizations"][0]
print(f"\nCheckpoint: |M|(T=1.5)={m_ord:.3f} (>0.8?), "
      f"|M|(T=3.5)={m_dis:.3f} (<0.2?)")
assert m_ord > 0.8 and m_dis < 0.2, "Simulator failed the ordered/disordered check!"
print("PASS — ordered phase magnetized, disordered phase not.")


## A.2 — Generate the labeled dataset

We scan $T \in [1.0, 3.5]$ at 50 temperatures, drawing 100 decorrelated
configurations per temperature on an $L=40$ lattice — **5000 labeled images**
total. Label $1$ = ordered ($T<T_c$), label $0$ = disordered ($T>T_c$).

> Heavy cell (~1–2 min on Colab CPU). Leave it running while you read ahead.

In [ ]:
from tqdm import tqdm

L = 40
TC = 2.0 / np.log(1 + np.sqrt(2))      # exact Onsager Tc
N_TEMPS = 50
T_MIN, T_MAX = 1.0, 3.5
N_SAMPLES_PER_T = 100
N_EQUILIB = 2000
SAMPLE_INTERVAL = 10

print(f"2D Ising data: {L}x{L} lattice, exact Tc = {TC:.4f}")
temperatures_grid = np.linspace(T_MIN, T_MAX, N_TEMPS)

all_configs, all_temps, all_labels, all_mags = [], [], [], []
for T in tqdm(temperatures_grid, desc="Generating"):
    n_total = N_EQUILIB + N_SAMPLES_PER_T * SAMPLE_INTERVAL
    res = run_simulation(
        L=L, T=T, n_sweeps=n_total, n_equilib=N_EQUILIB,
        n_samples=N_SAMPLES_PER_T, sample_interval=SAMPLE_INTERVAL,
        seed=42 + int(T * 1000),
    )
    for cfg, mag in zip(res["configs"], res["magnetizations"]):
        all_configs.append(cfg)
        all_temps.append(T)
        all_labels.append(1 if T < TC else 0)   # 1=ordered, 0=disordered
        all_mags.append(mag)

configs = np.array(all_configs, dtype=np.int8)
temperatures = np.array(all_temps, dtype=np.float32)
labels = np.array(all_labels, dtype=np.int64)
magnetizations = np.array(all_mags, dtype=np.float32)

print(f"\nconfigs shape   : {configs.shape}")          # (5000, 40, 40)
print(f"labels          : {np.sum(labels==1)} ordered, "
      f"{np.sum(labels==0)} disordered")


### Checkpoint A.2 — sanity-check the data against physics

A magnetization-vs-temperature curve should show a sharp drop right at
Onsager's $T_c$. We also verify the two-sided bound:
$\langle|M|\rangle > 0.8$ for $T<2.0$ and $< 0.2$ for $T>2.5$.

In [ ]:
unique_temps = np.unique(temperatures)
mean_mag = np.array([magnetizations[temperatures == T].mean() for T in unique_temps])

plt.figure(figsize=(7, 4.5))
plt.plot(unique_temps, mean_mag, "bo-", ms=4)
plt.axvline(TC, color="r", ls="--", label=f"$T_c$ = {TC:.3f} (Onsager)")
plt.xlabel("Temperature $T$ ($J/k_B$)")
plt.ylabel(r"$\langle |M| \rangle$")
plt.title("2D Ising: magnetization vs temperature")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

low_T = magnetizations[temperatures < 2.0].mean()
high_T = magnetizations[temperatures > 2.5].mean()
print(f"<|M|> for T<2.0 : {low_T:.3f}  (expect > 0.8)")
print(f"<|M|> for T>2.5 : {high_T:.3f}  (expect < 0.2)")


Let's also *look* at the raw images the network will see — the texture
visibly coarsens as we cross $T_c$.

In [ ]:
sample_temps = [1.0, 1.5, 2.0, 2.27, 2.5, 3.5]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, T_target in zip(axes.flat, sample_temps):
    idx = np.argmin(np.abs(unique_temps - T_target))
    T_actual = unique_temps[idx]
    cfg = configs[temperatures == T_actual][0]
    ax.imshow(cfg, cmap="coolwarm", vmin=-1, vmax=1, interpolation="nearest")
    phase = "ordered" if T_actual < TC else "disordered"
    ax.set_title(f"$T$ = {T_actual:.2f} ({phase})")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Sample Ising configurations")
plt.tight_layout(); plt.show()


## A.3 — Baseline: a Fully-Connected Network (FNN)

First a network with **no spatial structure**: flatten the $40\times40$ lattice
to a 1600-dim vector and feed it to an MLP ($1600 \to 128 \to 64 \to 2$). This
is our baseline — it must throw away the lattice geometry.

In [ ]:
from sklearn.model_selection import train_test_split


class IsingFNN(nn.Module):
    def __init__(self, input_dim=1600):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


def train_classifier(model, X_train, y_train, X_test, y_test,
                     n_epochs=15, batch_size=128, lr=3e-4):
    """Generic train loop; returns (train_accs, test_accs)."""
    model = model.to(device)
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
        batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(
        TensorDataset(torch.tensor(X_test), torch.tensor(y_test)),
        batch_size=batch_size)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params:,} parameters")

    train_accs, test_accs = [], []
    for epoch in range(n_epochs):
        model.train()
        correct = total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            correct += out.argmax(1).eq(yb).sum().item(); total += yb.size(0)
        train_accs.append(correct / total)

        model.eval(); correct = total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                correct += model(xb).argmax(1).eq(yb).sum().item(); total += yb.size(0)
        test_accs.append(correct / total)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  epoch {epoch+1:2d}: train_acc={train_accs[-1]:.3f}  "
                  f"test_acc={test_accs[-1]:.3f}")
    return train_accs, test_accs


# Flatten for the FNN
configs_f = configs.astype(np.float32)
N = configs_f.shape[0]
X = configs_f.reshape(N, -1)                       # (N, 1600)
y = labels
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"X_train shape: {X_train.shape},  X_test shape: {X_test.shape}")

fnn = IsingFNN()
fnn_train_acc, fnn_test_acc = train_classifier(fnn, X_train, y_train, X_test, y_test)
print(f"\nFNN final test accuracy: {fnn_test_acc[-1]:.4f}  (target >= 0.90)")


## A.4 — The CNN — exploiting spatial structure

Now keep the $40\times40$ **image** and use convolutions. The inductive bias —
**locality** and **weight sharing** — matches the physics: spin–spin
correlations are local, and the rule that produced the texture is the same
everywhere on the lattice (translation symmetry).

> **Connecting to L06.** In Lecture 06 we were careful to call this property
> **translation *equivariance*** (shift the input, the feature map shifts) — not
> "invariance". Convolution is equivariant by construction; the *pooling +
> classifier* head is what finally builds an (approximately) shift-*invariant*
> decision. Keep that distinction in mind — it returns in Mission B.

In [ ]:
class IsingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),   # ->(16,19,19)
            nn.Conv2d(16, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),  # ->(32, 8, 8)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64), nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Image format: (N, 1, 40, 40)
Xc = configs_f[:, np.newaxis, :, :]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"CNN input shape: {Xc_train.shape[1:]}")   # (1, 40, 40)

cnn = IsingCNN()
cnn_train_acc, cnn_test_acc = train_classifier(cnn, Xc_train, yc_train, Xc_test, yc_test)
print(f"\nCNN final test accuracy: {cnn_test_acc[-1]:.4f}  (target >= 0.92)")


### Checkpoint A.4 — FNN vs CNN

Both should exceed 90%. The CNN typically matches or beats the FNN with **far
fewer parameters**, because convolution bakes in the locality of the physics
instead of having to learn it. (On this clean toy data both saturate near
~99%, so the margin is small — the real payoff of the inductive bias shows up
on the *noisy* galaxy images in Mission B.)

In [ ]:
print(f"FNN test acc: {fnn_test_acc[-1]:.4f}")
print(f"CNN test acc: {cnn_test_acc[-1]:.4f}")
print(f"FNN params : {sum(p.numel() for p in fnn.parameters()):,}")
print(f"CNN params : {sum(p.numel() for p in cnn.parameters()):,}")


## A.5 — Rediscovering $T_c$ from the network

Here is the payoff. For every temperature, feed *all* its configurations to the
trained network and average the softmax probability $P(\text{ordered})$. Plot it
against $T$: it should fall like a sigmoid, crossing $0.5$ at the network's
estimate of the critical temperature. Compare with Onsager's exact
$T_c = 2.269$.

In [ ]:
def p_ordered_curve(model, use_cnn):
    """Average P(ordered) at each temperature."""
    model.eval()
    out = []
    with torch.no_grad():
        for T in unique_temps:
            cfgs = configs_f[temperatures == T]
            if use_cnn:
                x = torch.tensor(cfgs[:, np.newaxis, :, :]).to(device)
            else:
                x = torch.tensor(cfgs.reshape(cfgs.shape[0], -1)).to(device)
            probs = torch.softmax(model(x), dim=1)
            out.append(probs[:, 1].mean().item())   # class 1 = ordered
    return np.array(out)


def estimate_tc(temps, p):
    """Tc = temperature where P(ordered) crosses 0.5 (linear interp)."""
    for i in range(len(temps) - 1):
        if p[i] >= 0.5 and p[i + 1] < 0.5:
            t1, t2, p1, p2 = temps[i], temps[i + 1], p[i], p[i + 1]
            return t1 + (0.5 - p1) * (t2 - t1) / (p2 - p1)
    return np.nan


fnn_p = p_ordered_curve(fnn, use_cnn=False)
cnn_p = p_ordered_curve(cnn, use_cnn=True)
tc_fnn = estimate_tc(unique_temps, fnn_p)
tc_cnn = estimate_tc(unique_temps, cnn_p)

print(f"Exact (Onsager) : Tc = {TC:.4f}")
print(f"FNN estimate    : Tc = {tc_fnn:.4f}  (error = {abs(tc_fnn-TC):.4f})")
print(f"CNN estimate    : Tc = {tc_cnn:.4f}  (error = {abs(tc_cnn-TC):.4f})")

fig, ax = plt.subplots(figsize=(7.5, 5))
ax.axvspan(1.0, TC, alpha=0.10, color="blue", label="train label: ordered")
ax.axvspan(TC, 3.5, alpha=0.10, color="red", label="train label: disordered")
ax.plot(unique_temps, fnn_p, "bo-", ms=4, label=f"FNN ($T_c$={tc_fnn:.3f})")
ax.plot(unique_temps, cnn_p, "rs-", ms=4, label=f"CNN ($T_c$={tc_cnn:.3f})")
ax.axvline(TC, color="k", ls="--", alpha=0.7, label=f"exact $T_c$={TC:.3f}")
ax.axhline(0.5, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("Temperature $T$ ($J/k_B$)")
ax.set_ylabel(r"$P(\mathrm{ordered})$")
ax.set_title("Neural-network phase classification: $P$(ordered) vs $T$")
ax.legend(loc="center right", fontsize=9); ax.grid(alpha=0.3)
ax.set_ylim(-0.05, 1.05); ax.set_xlim(1.0, 3.5)
plt.tight_layout(); plt.show()


### Checkpoint A.5 — did the network rediscover the transition?

Both estimates should satisfy $|T_c^{\text{est}} - 2.269| < 0.15$. The network
was **never told** where the transition is — it only saw "ordered"/"disordered"
labels — yet the crossover of its order parameter lands on Onsager's exact
value. That is the central, slightly magical, result of Carrasquilla & Melko.

In [ ]:
for name, tc in [("FNN", tc_fnn), ("CNN", tc_cnn)]:
    err = abs(tc - TC)
    status = "PASS" if err < 0.15 else "CHECK"
    print(f"{name}: Tc_est={tc:.3f}, |Tc_est - 2.269|={err:.3f}  [{status}]")


---

# Mission B — Galaxy morphology from real telescope images

**Paper:** Dieleman, Willett & Dambre, *Rotation-invariant CNNs for galaxy
morphology prediction*, *MNRAS* **450**, 1441 (2015).

## The physics first

Galaxies come in broad morphological classes — **elliptical** (smooth,
featureless blobs) and **spiral** (disks with arms). Historically humans
classified them by eye (the *Galaxy Zoo* citizen-science project). Here we let a
CNN do it from **real observed images**.

> ### Data label: **Real measured data**
> **Galaxy10 DECals** — cutouts photographed by the **Dark Energy Camera
> (DECam)** on the 4-m Blanco telescope. We use a balanced binary subset:
> 2500 elliptical + 2500 spiral, resized to $64\times64$ RGB (5000 images,
> a few tens of MB). This is the first time in the course a network sees data a
> physical instrument actually recorded — complete with noise, seeing, and
> ambiguous edge cases.

**A symmetry to exploit.** A galaxy's morphological class does not depend on how
it happens to be oriented on the sky. There is **no preferred orientation** in
the observation. That physical fact justifies **augmenting** the training set
with flipped/rotated copies — we return to this in the Step-2 checkpoint.


## B.1 — Download and explore the data

The instructor has pre-extracted a balanced subset (`galaxy_subset.npz`) hosted
on Google Drive. The cell below downloads it with `gdown` and inspects it.

> Network cell — runs in Colab. If `gdown` is missing, the install cell at the
> top adds it. (Instructors: `prepare_subset.py` in the solution folder builds
> this file from the full 2.5 GB Galaxy10 DECals release.)

In [ ]:
import os

# Instructor-hosted balanced subset (2500 elliptical + 2500 spiral, 64x64 RGB).
GDRIVE_FILE_ID = "1c_WZle0IJiGzojZXDFMWMA_ffsMP4OYS"
OUTPUT_FILE = "galaxy_subset.npz"

if not os.path.exists(OUTPUT_FILE):
    import gdown
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}",
                   OUTPUT_FILE, quiet=False)

data = np.load(OUTPUT_FILE)
images = data["images"]    # (5000, 64, 64, 3) uint8
gal_labels = data["labels"]  # (5000,)  0=elliptical, 1=spiral
print(f"images shape : {images.shape}  dtype={images.dtype}  "
      f"range=[{images.min()}, {images.max()}]")
print(f"labels       : {(gal_labels==0).sum()} elliptical, "
      f"{(gal_labels==1).sum()} spiral")


### Checkpoint B.1 — can *you* tell them apart?

Look at 8 elliptical (top) and 8 spiral (bottom) galaxies. Ellipticals are
smooth blobs; spirals show disk/arm structure. Note already how some are blurry
or nearly edge-on — those will be the model's hard cases later.

In [ ]:
CLASS_NAMES = ["Elliptical", "Spiral"]
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for row, cls in enumerate([0, 1]):
    idxs = np.where(gal_labels == cls)[0][:8]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(images[idx])
        axes[row, col].set_title(CLASS_NAMES[cls], fontsize=9)
        axes[row, col].axis("off")
fig.suptitle("Galaxy10 DECals subset — real DECam images")
plt.tight_layout(); plt.show()


## B.2 — From-scratch CNN

A standard 3-conv-block CNN ($3\times64\times64 \to 2$) with dropout, trained
from random initialization. We split 70/15/15 (train/val/test) and apply
`RandomHorizontalFlip` as physically-motivated augmentation.

We also wrap the train loop into a small helper that returns the **wall-clock
training time** — we'll need it for the Mission B+ comparison.

In [ ]:
import time
import torchvision.transforms.v2 as T


class GalaxyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),    # ->(32,31,31)
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),   # ->(64,14,14)
            nn.Conv2d(64, 128, 3), nn.ReLU(), nn.MaxPool2d(2),  # ->(128,6,6)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Tensors: (N, C, H, W) in [0, 1]
X_gal = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
y_gal = torch.tensor(gal_labels, dtype=torch.long)
print(f"X_gal shape: {X_gal.shape}")

n_total = len(X_gal)
n_train = int(0.7 * n_total); n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val
gen = torch.Generator().manual_seed(SEED)
dataset = TensorDataset(X_gal, y_gal)
train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test], generator=gen)
print(f"Train {n_train}, Val {n_val}, Test {n_test}")

BATCH = 128
train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH)
test_loader  = DataLoader(test_set,  batch_size=BATCH)


def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += model(xb).argmax(1).eq(yb).sum().item(); total += yb.size(0)
    return correct / total


def train_galaxy(model, n_epochs=20, lr=3e-4, augment=None, label=""):
    """Train; return dict with histories, test_acc, train_time, trainable params."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr, weight_decay=1e-4)
    n_trainable = sum(p.numel() for p in params)
    print(f"[{label}] trainable params: {n_trainable:,}")

    train_accs, val_accs = [], []
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train(); correct = total = 0
        for xb, yb in train_loader:
            if augment is not None:
                xb = augment(xb)
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
            correct += model(xb).argmax(1).eq(yb).sum().item(); total += yb.size(0)
        train_accs.append(correct / total)
        val_accs.append(evaluate(model, val_loader))
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  epoch {epoch+1:2d}: train_acc={train_accs[-1]:.3f}  "
                  f"val_acc={val_accs[-1]:.3f}")
    train_time = time.time() - t0
    test_acc = evaluate(model, test_loader)
    print(f"[{label}] test_acc={test_acc:.4f}  train_time={train_time:.1f}s")
    return {"train_accs": train_accs, "val_accs": val_accs,
            "test_acc": test_acc, "train_time": train_time,
            "n_trainable": n_trainable}


augment = T.RandomHorizontalFlip()   # galaxies have no preferred orientation

# Keep an explicit handle on the trained model — we reuse it for evaluation in B.3.
torch.manual_seed(SEED)
galaxy_cnn = GalaxyCNN()
scratch = train_galaxy(galaxy_cnn, n_epochs=20, augment=augment, label="from-scratch")


### Checkpoint B.2 — overfitting and the physics of augmentation

**Numerical checks:**
- Test accuracy $\ge 85\%$.
- Overfitting gap: $\text{train\_acc} - \text{val\_acc} < 15\%$.

**Physics note — why `RandomHorizontalFlip`?** (connecting back to L06)

A galaxy's morphology is a *physical* property; how it happens to be oriented on
the detector is not. So a mirror-flipped image of a spiral is *still the same
spiral* — the label is invariant under reflection. Augmenting with flips tells
the network this symmetry for free, enlarging the effective dataset along a
direction we *know* leaves the answer unchanged.

> **Careful with the word.** This is **flip (reflection) symmetry of the
> *label*** — it is *not* the translation **equivariance** of the convolution
> itself (the property we corrected the L06 wording on: shifting the input
> shifts the feature map, it does **not** leave it invariant). Equivariance is a
> structural property of the *layer*; the flip augmentation here is a statement
> about a *symmetry of the labeling function* that we inject through the data.
> Both are "symmetry," but they act at different places — keep them distinct.

In [ ]:
gap = scratch["train_accs"][-1] - scratch["val_accs"][-1]
print(f"from-scratch test_acc      : {scratch['test_acc']:.4f}  (>= 0.85?)")
print(f"overfitting gap (train-val): {gap:.4f}  (< 0.15?)")

ep = range(1, len(scratch["train_accs"]) + 1)
plt.figure(figsize=(7, 4.5))
plt.plot(ep, scratch["train_accs"], "b-", label="train")
plt.plot(ep, scratch["val_accs"], "r-", label="val")
plt.xlabel("epoch"); plt.ylabel("accuracy")
plt.title("From-scratch GalaxyCNN")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## B.3 — Evaluate: confusion matrix and the hard cases

A confusion matrix shows *which* class the model confuses. Then we look at the
model's most confident **mistakes** — these reveal the physical edge cases. We
evaluate the `galaxy_cnn` we trained in B.2 (we kept a handle on it).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report


def collect_predictions(model, loader):
    model.eval(); preds, trues, probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            p = torch.softmax(model(xb), dim=1).cpu().numpy()
            preds.extend(p.argmax(1)); trues.extend(yb.numpy()); probs.extend(p)
    return np.array(preds), np.array(trues), np.array(probs)


preds, trues, probs = collect_predictions(galaxy_cnn, test_loader)
print(classification_report(trues, preds, target_names=CLASS_NAMES))

cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
for i in range(2):
    for j in range(2):
        color = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=15, color=color)
fig.colorbar(im); plt.tight_layout(); plt.show()


### Checkpoint B.3 — read the mistakes

The confusion matrix should be **diagonal-dominant**. Now look at the most
confident misclassifications below and describe the pattern: typically the model
trips on **edge-on spirals** (which look like smooth cigars, hence "elliptical")
and **blurry / low-S/N** images where the arm structure is washed out. That is a
physically sensible failure mode, not random error.

In [ ]:
# Indices into the original array for the test split
test_idx = np.array(test_set.indices)
test_imgs = images[test_idx]

wrong = np.where(preds != trues)[0]
wrong = wrong[np.argsort(-probs[wrong].max(1))]    # most confident mistakes first
n_show = min(6, len(wrong))
if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3.2))
    if n_show == 1:
        axes = [axes]
    for k, idx in enumerate(wrong[:n_show]):
        axes[k].imshow(test_imgs[idx])
        conf = probs[idx].max() * 100
        axes[k].set_title(f"pred {CLASS_NAMES[preds[idx]]} ({conf:.0f}%)\n"
                          f"true {CLASS_NAMES[trues[idx]]}", fontsize=9, color="red")
        axes[k].axis("off")
    fig.suptitle("Most confident mistakes — edge-on / blurry?")
    plt.tight_layout(); plt.show()
else:
    print("No misclassifications in the test set!")


---

# Mission B+ — Transfer learning with a pretrained ResNet-18 (NEW)

So far every weight we trained started from random noise. But somebody already
spent enormous compute teaching a **ResNet-18** to see *general* visual
features — edges, blobs, textures — by training it on **ImageNet** (1.2M natural
images). Those low-level features are not specific to cats and cars; **edges and
blobs are edges and blobs**, galaxies included.

**Transfer learning** reuses that knowledge:

1. Load `torchvision.models.resnet18(weights=...)` — pretrained on ImageNet.
2. **Freeze** the whole backbone (`requires_grad = False`) — we keep its
   features fixed.
3. Replace the final classifier `fc` ($512 \to 1000$ ImageNet classes) with a
   tiny new head ($512 \to 2$: elliptical / spiral).
4. **Fine-tune only the head** on our 5000 galaxy images.

> **Physicist's analogy.** This is like starting a variational calculation from
> a known good trial wavefunction instead of a random one — most of the
> structure is already right, so you only relax the few parameters that matter.
> Freezing the backbone means we never compute backbone gradients, so each epoch
> is **much cheaper** despite the model being ~11M parameters.

This is the **first time in the course** we fine-tune a pretrained model.

## B+.1 — Load, freeze, and re-head ResNet-18

ImageNet models expect 3-channel images normalized with ImageNet statistics. We
resize our $64\times64$ galaxies to $224\times224$ and apply that normalization.
Then we freeze everything and swap the head.

> Network cell (downloads ~45 MB of pretrained weights on first call) — runs in
> Colab.

In [ ]:
import torchvision
import torchvision.transforms.v2 as T

# ImageNet preprocessing: resize -> normalize with ImageNet mean/std.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
resnet_preprocess = T.Compose([
    T.Resize((224, 224), antialias=True),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Load ImageNet-pretrained ResNet-18.
# (Modern API: weights=...; the legacy pretrained=True is equivalent.)
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
resnet = torchvision.models.resnet18(weights=weights)

# 1) Freeze the entire backbone.
for p in resnet.parameters():
    p.requires_grad = False

# 2) Replace the head: fc is Linear(512 -> 1000); we want 512 -> 2.
in_features = resnet.fc.in_features          # 512
resnet.fc = nn.Linear(in_features, 2)        # new head: requires_grad=True by default
resnet = resnet.to(device)

n_trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
n_total_p = sum(p.numel() for p in resnet.parameters())
print(f"ResNet-18 total params     : {n_total_p:,}")
print(f"ResNet-18 trainable params : {n_trainable:,}  (head only)")

# ---- Optional alternative: a Vision Transformer via timm (one line) ----
# import timm
# vit = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=2)


### Checkpoint B+.1 — confirm the backbone is frozen

Only the new `fc` head should be trainable. For a $512\to2$ linear layer:
$512\times2$ weights $+\,2$ biases $= \mathbf{1026}$ parameters. Verify
$1026 \le \text{trainable} \le 2050$ (i.e. the **fc layer only**).

In [ ]:
assert 1026 <= n_trainable <= 2050, (
    f"Expected only the fc head to be trainable (~1026 params), got {n_trainable}")
print(f"PASS — backbone frozen, only the {n_trainable}-param head will train.")


## B+.2 — Fine-tune the head

We reuse the **same** train/val/test split as Mission B (so the comparison is
fair), but now each image is passed through `resnet_preprocess` before the
network. Because the backbone is frozen, this converges **fast**.

> Heavy-ish cell (~2 min on Colab CPU; seconds on GPU). The frozen backbone
> means no backbone gradients — that is the whole speed advantage.

In [ ]:
def train_resnet(model, n_epochs=20, lr=1e-3, preprocess=None, label="resnet18-ft"):
    """Fine-tune with ImageNet preprocessing; mirrors train_galaxy's outputs."""
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=lr)
    n_trainable = sum(p.numel() for p in params)
    print(f"[{label}] trainable params: {n_trainable:,}")

    def run_eval(loader):
        model.eval(); correct = total = 0
        with torch.no_grad():
            for xb, yb in loader:
                xb = preprocess(xb).to(device); yb = yb.to(device)
                correct += model(xb).argmax(1).eq(yb).sum().item(); total += yb.size(0)
        return correct / total

    train_accs, val_accs = [], []
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train(); correct = total = 0
        for xb, yb in train_loader:
            xb = augment(xb)                  # same flip augmentation
            xb = preprocess(xb).to(device); yb = yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            correct += out.argmax(1).eq(yb).sum().item(); total += yb.size(0)
        train_accs.append(correct / total)
        val_accs.append(run_eval(val_loader))
        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  epoch {epoch+1:2d}: train_acc={train_accs[-1]:.3f}  "
                  f"val_acc={val_accs[-1]:.3f}")
    train_time = time.time() - t0
    test_acc = run_eval(test_loader)
    print(f"[{label}] test_acc={test_acc:.4f}  train_time={train_time:.1f}s")
    return {"train_accs": train_accs, "val_accs": val_accs,
            "test_acc": test_acc, "train_time": train_time,
            "n_trainable": n_trainable}


finetune = train_resnet(resnet, n_epochs=20, preprocess=resnet_preprocess)


### Checkpoint B+.2 — fast convergence of a frozen backbone

A frozen pretrained backbone should converge **fast**: $\text{val\_acc} \ge
88\%$ by **epoch 5** is the target. Because the features are already meaningful,
the tiny head only has to learn a linear boundary in a good feature space.

In [ ]:
val5 = finetune["val_accs"][4] if len(finetune["val_accs"]) >= 5 else finetune["val_accs"][-1]
print(f"ResNet-18 fine-tune val_acc @ epoch 5: {val5:.4f}  (target >= 0.88)")
print("PASS" if val5 >= 0.88 else "CHECK")


## B+.3 — Head-to-head: from-scratch vs fine-tuned

Now the punchline. Same data, same split, same augmentation — two very different
training strategies. Compare three numbers: **test accuracy**, **total training
time**, and **number of trainable parameters**.

**What to expect:**
- ResNet-18 fine-tune: test acc $\gtrsim 94\%$
- From-scratch GalaxyCNN: test acc $\sim 92\%$
- The fine-tuned model trains in roughly **half the time** (frozen backbone =
  no backbone gradients) while updating **far fewer** parameters (~1k vs ~2M).

In [ ]:
def fmt(x):
    return f"{x:.4f}" if isinstance(x, float) else str(x)

rows = [
    ("from-scratch GalaxyCNN", scratch),
    ("ResNet-18 (frozen, fine-tuned)", finetune),
]
print(f"{'model':<32}{'test_acc':>10}{'train_time(s)':>16}{'trainable params':>20}")
print("-" * 78)
for name, r in rows:
    print(f"{name:<32}{r['test_acc']:>10.4f}{r['train_time']:>16.1f}"
          f"{r['n_trainable']:>20,}")

print("\nSpeedup (scratch_time / finetune_time): "
      f"{scratch['train_time'] / finetune['train_time']:.2f}x")
print(f"Accuracy delta (finetune - scratch): "
      f"{finetune['test_acc'] - scratch['test_acc']:+.4f}")

# Learning-curve overlay
plt.figure(figsize=(7.5, 4.5))
plt.plot(range(1, len(scratch["val_accs"]) + 1), scratch["val_accs"],
         "b-o", ms=3, label="from-scratch (val)")
plt.plot(range(1, len(finetune["val_accs"]) + 1), finetune["val_accs"],
         "r-s", ms=3, label="ResNet-18 fine-tune (val)")
plt.axhline(0.88, color="gray", ls=":", alpha=0.6, label="0.88 target")
plt.xlabel("epoch"); plt.ylabel("validation accuracy")
plt.title("Transfer learning vs training from scratch")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


### Checkpoint B+.3 — read the comparison table

All three numbers must print for **both** models: `test_acc`,
`total_train_time(s)`, and `trainable params`. The story to take away:

- Pretrained features give you a **head start** — comparable or higher accuracy
  with orders of magnitude fewer trainable parameters.
- Freezing the backbone makes each epoch **cheaper**, so fine-tuning is *both*
  more data-efficient *and* faster here.
- This is the standard modern recipe whenever you have a **small** dataset (5000
  images is small!) and a relevant pretrained model — extremely common in
  observational astronomy, microscopy, and medical imaging.

---

# Wrap-up and exercises

**What you did today**

- **Mission A:** generated Boltzmann-sampled Ising configurations, trained an
  FNN and a CNN, and watched the network **rediscover Onsager's $T_c$** from
  labels alone (Carrasquilla & Melko).
- **Mission B:** classified **real DECam galaxy images** with a from-scratch
  CNN, and learned to read its mistakes physically (edge-on spirals, blur).
- **Mission B+:** **fine-tuned an ImageNet-pretrained ResNet-18** by freezing
  the backbone and retraining only a 2-class head — your first transfer-learning
  experiment — and benchmarked it against the from-scratch model.

**The big idea.** Convolutions encode the *locality* and *translation
equivariance* of physical fields; symmetry-aware **augmentation** injects label
symmetries (flips) we know a priori; and **transfer learning** lets us inherit
generic visual features instead of relearning edges from scratch. All three are
ways of putting *physics / prior knowledge* into the model rather than hoping the
data alone supplies it.

### Exercises

1. **Finite-size scaling (Mission A).** Regenerate Ising data at $L=20, 40, 60$.
   Does the network's $T_c$ estimate sharpen (steeper $P$(ordered) crossover) as
   $L$ grows? Relate this to finite-size rounding of the true transition.
2. **Equivariance check (Mission A).** Take a trained `IsingCNN`, translate an
   input configuration by a few pixels (with periodic wrap), and confirm the
   feature maps shift accordingly while $P$(ordered) is (approximately)
   unchanged. This makes the L06 equivariance-vs-invariance distinction
   concrete.
3. **Rotation augmentation (Mission B).** Add `T.RandomRotation(degrees=180)` to
   the from-scratch CNN's augmentation. Galaxies have full rotational symmetry —
   does test accuracy improve over flip-only?
4. **Unfreeze and fine-tune deeper (Mission B+).** Unfreeze ResNet-18's last
   block (`layer4`) in addition to the head, use a smaller learning rate
   (e.g. `1e-4`), and see whether accuracy rises — at what cost in training time?
5. **Try a ViT (Mission B+).** Uncomment the `timm.create_model(...)` line, load
   a pretrained `vit_tiny_patch16_224`, and compare it head-to-head with
   ResNet-18 on the same split.

### References

- J. Carrasquilla & R. G. Melko, *Machine learning phases of matter*,
  **Nature Physics 13, 431 (2017)**.
- S. Dieleman, K. W. Willett & J. Dambre, *Rotation-invariant CNNs for galaxy
  morphology prediction*, **MNRAS 450, 1441 (2015)**.
- K. He et al., *Deep Residual Learning for Image Recognition (ResNet)*,
  **CVPR 2016**.
- Galaxy10 DECals dataset: https://astronn.readthedocs.io/en/latest/galaxy10.html
